# CNN-BiLSTM-Attention — Mitral ECG (Tiffany Flat 3-class)

Architecture: Multi-scale CNN → BiLSTM → Temporal Soft Attention → Classifier  
~0.9M params. Designed for small ECG datasets — multi-scale receptive fields + sequence modeling + attention.

Same training protocol as `mitral_inceptiontime_tiffany_flat`: 5-fold CV, heldout evaluation, class-weighted CE.

In [ ]:
from __future__ import annotations

import random
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
PROJECT_ROOT   = Path("/kaggle/input/datasets/pascalhafidzfajri/mlcv-mitral")
OUT_DIR        = Path(".")
NPZ_DIR        = PROJECT_ROOT / "mitral_images" / "mitral_images"
SPLIT_CSV_PATH = PROJECT_ROOT / "data_split_3level_exp10.csv"
ARTIFACT_DIR   = OUT_DIR / "dl_artifacts" / "cnn_bilstm_attention_tiffany_flat"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES         = ["0_control", "1_significant_mr", "2_significant_ms"]
FLAT_LABEL_TO_CLASS = {"Kontrol": 0, "MR": 1, "MS": 2}
LEAD_ORDER          = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]

DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IN_CHANNELS    = len(LEAD_ORDER)  # 12
BATCH_SIZE     = 32
EPOCHS         = 60
PATIENCE       = 15
LEARNING_RATE  = 1e-3
MIN_LR         = 1e-5
WEIGHT_DECAY   = 1e-4
GRAD_CLIP_NORM = 5.0
NUM_WORKERS    = 0
PRELOAD_TO_RAM = True
SKIP_COMPLETED_FOLDS = True

AUGMENTATION_CONFIG = {
    "noise_prob": 0.50, "noise_std": 0.01,
    "scale_prob": 0.50, "scale_range": (0.95, 1.05),
    "time_mask_prob": 0.35, "time_mask_max_frac": 0.03,
    "lead_dropout_prob": 0.15, "max_dropped_leads": 1,
}

print("Device:", DEVICE)
print("Project root:", PROJECT_ROOT)
print("Artifact dir:", ARTIFACT_DIR)

## 1. Data loading

In [ ]:
NPZ_RAM_CACHE: dict = {}

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def infer_original_code(stem: str) -> str:
    stem = str(stem).strip().upper()
    for prefix in ("BA", "BB", "A", "B", "C", "D", "E"):
        if stem.startswith(prefix):
            return prefix
    return "UNKNOWN"


def normalize_per_sample_per_lead(signal: np.ndarray) -> np.ndarray:
    finite = np.isfinite(signal)
    valid_counts = finite.sum(axis=0, keepdims=True)
    valid_counts_f = np.maximum(valid_counts.astype(np.float32), 1.0)
    signal_sum = np.where(finite, signal, 0.0).sum(axis=0, keepdims=True).astype(np.float32)
    sample_mean = signal_sum / valid_counts_f
    centered = np.where(finite, signal - sample_mean, 0.0).astype(np.float32)
    signal_var = (centered ** 2).sum(axis=0, keepdims=True) / valid_counts_f
    sample_std = np.sqrt(np.maximum(signal_var, 1e-6)).astype(np.float32)
    all_invalid = valid_counts == 0
    sample_mean = np.where(all_invalid, 0.0, sample_mean).astype(np.float32)
    sample_std  = np.where(all_invalid, 1.0, sample_std).astype(np.float32)
    signal = (np.where(finite, signal, sample_mean) - sample_mean) / sample_std
    return np.nan_to_num(signal, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def augment_signal_channels(signal_ch: np.ndarray, rng: np.random.Generator, cfg: dict) -> np.ndarray:
    x = signal_ch.copy()
    n_leads, n_steps = x.shape
    if rng.random() < cfg["scale_prob"]:
        lo, hi = cfg["scale_range"]
        x *= rng.uniform(lo, hi, size=(n_leads, 1)).astype(np.float32)
    if rng.random() < cfg["noise_prob"] and cfg["noise_std"] > 0:
        x += rng.normal(0.0, cfg["noise_std"], size=x.shape).astype(np.float32)
    if rng.random() < cfg["time_mask_prob"] and cfg["time_mask_max_frac"] > 0:
        max_width = max(1, int(n_steps * cfg["time_mask_max_frac"]))
        width = int(rng.integers(1, max_width + 1))
        start = int(rng.integers(0, max(1, n_steps - width + 1)))
        x[:, start:start + width] = 0.0
    if rng.random() < cfg["lead_dropout_prob"] and cfg["max_dropped_leads"] > 0:
        n_drop = int(rng.integers(1, cfg["max_dropped_leads"] + 1))
        leads = rng.choice(n_leads, size=n_drop, replace=False)
        x[leads, :] = 0.0
    return x.astype(np.float32)


def load_npz_arrays(npz_path: str):
    with np.load(npz_path, allow_pickle=False) as d:
        return d["signal_matrix"].astype(np.float32), d["overlap_mask"].astype(np.uint8), d["interpolated_mask"].astype(np.uint8)


def get_npz_arrays(row: pd.Series):
    key = str(row["stem"])
    if not PRELOAD_TO_RAM:
        return load_npz_arrays(row["npz_path"])
    if key not in NPZ_RAM_CACHE:
        NPZ_RAM_CACHE[key] = load_npz_arrays(row["npz_path"])
    return NPZ_RAM_CACHE[key]


def preload_npz_frame(frame: pd.DataFrame, desc: str) -> None:
    if not PRELOAD_TO_RAM:
        return
    pending = [
        (str(r["stem"]), str(r["npz_path"]))
        for _, r in frame.drop_duplicates("stem").iterrows()
        if str(r["stem"]) not in NPZ_RAM_CACHE
    ]
    if not pending:
        return
    def load_one(item):
        stem, path = item
        return stem, load_npz_arrays(path)
    with ThreadPoolExecutor(max_workers=4) as ex:
        for stem, arrays in tqdm(
            (f.result() for f in as_completed(ex.submit(load_one, item) for item in pending)),
            total=len(pending), desc=f"preload {desc}", leave=False
        ):
            NPZ_RAM_CACHE[stem] = arrays


class MitralECGDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, augment: bool = False, seed: int = SEED):
        self.frame = frame.reset_index(drop=True)
        self.augment = augment
        self.seed = seed
        preload_npz_frame(self.frame, "ECG")

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        signal, overlap, _ = get_npz_arrays(row)
        signal[overlap.astype(bool)] = 0.0
        signal_ch = normalize_per_sample_per_lead(signal).T  # (12, T)
        if self.augment:
            rng = np.random.default_rng(self.seed + idx * 1009 + random.randint(0, 1_000_000))
            signal_ch = augment_signal_channels(signal_ch, rng, AUGMENTATION_CONFIG)
        return (
            torch.from_numpy(signal_ch.astype(np.float32)),
            torch.tensor(int(row["class_id"]), dtype=torch.long),
            str(row["stem"]),
        )


def load_tiffany_split():
    split = pd.read_csv(SPLIT_CSV_PATH)
    split["stem"] = split["filename"].map(lambda x: Path(str(x)).stem)
    split["original_code"] = split["stem"].map(infer_original_code)
    split["npz_path"] = split["stem"].map(lambda s: str(NPZ_DIR / f"{s}.npz"))
    split["npz_exists"] = split["npz_path"].map(lambda p: Path(p).exists())
    available = split[split["npz_exists"]].copy().reset_index(drop=True)
    available["class_id"] = available["flat_label"].map(FLAT_LABEL_TO_CLASS)
    print(f"Available samples: {len(available)} / {len(split)} (missing NPZ: {len(split)-len(available)})")
    train_val = available[available["isHeldOut"] == 0].copy().reset_index(drop=True)
    heldout   = available[available["isHeldOut"] == 1].copy().reset_index(drop=True)
    print(f"Train/val: {len(train_val)}, Heldout: {len(heldout)}")
    return train_val, heldout


def make_loader(frame, batch_size, shuffle, augment, seed):
    ds = MitralECGDataset(frame, augment=augment, seed=seed)
    g = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=(DEVICE.type == "cuda"), generator=g)


set_seed(SEED)
train_val_df, heldout_df = load_tiffany_split()
print("\nClass distribution (train/val):")
print(train_val_df["flat_label"].value_counts().to_string())
print("\nClass distribution (heldout):")
print(heldout_df["flat_label"].value_counts().to_string())

## 2. CNN-BiLSTM-Attention architecture

```
Input: (B, 12, T)
  ├─ Branch-S: Conv1d(12→32, k=7) × 2 + MaxPool(4)  ─┐
  ├─ Branch-M: Conv1d(12→32, k=15) × 2 + MaxPool(4) ─├─ cat → (B, 96, T/4)
  └─ Branch-L: Conv1d(12→32, k=31) × 2 + MaxPool(4) ─┘
  → Conv1d(96→128, k=1)  projection
  → BiLSTM(128, 256)     → (B, T/4, 512)
  → Temporal attention   → (B, 512)
  → Dropout + Linear     → (B, 3)
```
~0.9M parameters.

In [ ]:
class CNNBranch(nn.Module):
    """Two-layer 1D CNN branch with given kernel size."""
    def __init__(self, in_ch: int, out_ch: int, kernel: int, pool: int = 4):
        super().__init__()
        p = kernel // 2
        self.net = nn.Sequential(
            nn.Conv1d(in_ch,  out_ch, kernel, padding=p, bias=False),
            nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True),
            nn.Conv1d(out_ch, out_ch, kernel, padding=p, bias=False),
            nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True),
            nn.MaxPool1d(pool),
        )

    def forward(self, x):
        return self.net(x)


class TemporalAttention(nn.Module):
    """Additive soft attention over the time dimension."""
    def __init__(self, hidden_size: int):
        super().__init__()
        self.score = nn.Linear(hidden_size, 1)

    def forward(self, x):  # x: (B, T, H)
        w = torch.softmax(self.score(x), dim=1)  # (B, T, 1)
        return (x * w).sum(dim=1)                # (B, H)


class CNNBiLSTMAttention(nn.Module):
    """
    Multi-scale CNN → BiLSTM → Temporal Attention.
    Kernel sizes 7/15/31 capture short/medium/long ECG patterns.
    """
    def __init__(self, in_channels: int = 12, num_classes: int = 3,
                 branch_ch: int = 32, lstm_hidden: int = 256, dropout: float = 0.3):
        super().__init__()
        self.branch_s = CNNBranch(in_channels, branch_ch, kernel=7)
        self.branch_m = CNNBranch(in_channels, branch_ch, kernel=15)
        self.branch_l = CNNBranch(in_channels, branch_ch, kernel=31)

        feat_ch = branch_ch * 3  # 96
        self.proj = nn.Sequential(
            nn.Conv1d(feat_ch, 128, 1, bias=False),
            nn.BatchNorm1d(128), nn.ReLU(inplace=True),
        )

        self.bilstm = nn.LSTM(
            input_size=128, hidden_size=lstm_hidden,
            num_layers=1, batch_first=True,
            bidirectional=True, dropout=0.0,
        )

        self.attention = TemporalAttention(lstm_hidden * 2)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(lstm_hidden * 2, num_classes)

    def forward(self, x):  # x: (B, 12, T)
        f_s = self.branch_s(x)
        f_m = self.branch_m(x)
        f_l = self.branch_l(x)
        L = min(f_s.shape[-1], f_m.shape[-1], f_l.shape[-1])
        feat = torch.cat([f_s[..., :L], f_m[..., :L], f_l[..., :L]], dim=1)  # (B, 96, L)
        feat = self.proj(feat)              # (B, 128, L)
        feat = feat.transpose(1, 2)        # (B, L, 128)
        out, _ = self.bilstm(feat)         # (B, L, 512)
        ctx = self.attention(out)          # (B, 512)
        return self.head(self.drop(ctx))


# Parameter count
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

m = CNNBiLSTMAttention(IN_CHANNELS, len(CLASS_NAMES))
print(f"CNNBiLSTMAttention: {count_params(m):,} trainable parameters")
del m

## 3. Training utilities

In [ ]:
def stable_softmax(logits: np.ndarray) -> np.ndarray:
    x = np.asarray(logits, dtype=np.float64)
    x -= x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return (e / e.sum(axis=1, keepdims=True)).astype(np.float32)


def metrics_from_probs(y_true, probs, class_names, prefix=""):
    pred = probs.argmax(axis=1)
    labels = list(range(len(class_names)))
    recalls = recall_score(y_true, pred, labels=labels, average=None, zero_division=0)
    return {
        f"{prefix}accuracy":           accuracy_score(y_true, pred),
        f"{prefix}balanced_accuracy":  balanced_accuracy_score(y_true, pred),
        f"{prefix}macro_f1":           f1_score(y_true, pred, average="macro", zero_division=0),
        f"{prefix}weighted_f1":        f1_score(y_true, pred, average="weighted", zero_division=0),
        f"{prefix}macro_precision":    precision_score(y_true, pred, average="macro", zero_division=0),
        f"{prefix}macro_recall":       recall_score(y_true, pred, average="macro", zero_division=0),
        **{f"{prefix}recall_{c}": float(recalls[i]) for i, c in enumerate(class_names)},
        f"{prefix}min_class_recall":   float(recalls.min()),
        f"{prefix}macro_ovr_auc":      roc_auc_score(y_true, probs, multi_class="ovr", average="macro"),
    }


def class_weighted_ce(frame, label_col, n_classes):
    weights = compute_class_weight("balanced", classes=np.arange(n_classes), y=frame[label_col].values)
    return nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32).to(DEVICE))


def run_epoch(model, loader, criterion, optimizer, scaler, training, desc):
    model.train(training)
    total_loss, all_logits, all_labels, all_stems = 0.0, [], [], []
    ctx = torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda"))
    for x, y, stems in tqdm(loader, desc=desc, leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        with ctx:
            logits = model(x)
            loss = criterion(logits, y)
        if training:
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
        total_loss += loss.item() * len(y)
        all_logits.append(logits.detach().cpu().float().numpy())
        all_labels.append(y.cpu().numpy())
        all_stems.extend(stems)
    return (
        total_loss / len(loader.dataset),
        np.concatenate(all_labels),
        np.concatenate(all_logits),
        np.array(all_stems),
    )


def plot_confusion_matrix(y_true, y_pred, y_probs, fold, macro_f1, auc, class_names, val_score=None):
    n = len(class_names)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n)))
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(n + 2, n + 1))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    short = [c.split("_", 1)[-1] if "_" in c else c for c in class_names]
    ax.set_xticks(range(n), short, rotation=30, ha="right")
    ax.set_yticks(range(n), short)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{cm[i,j]}\n{cm_norm[i,j]:.2f}",
                    ha="center", va="center", fontsize=9,
                    color="white" if cm_norm[i,j] > 0.6 else "black")
    title = f"Fold {fold} Heldout | F1={macro_f1:.3f}  AUC={auc:.3f}"
    if val_score is not None:
        title += f"  val={val_score:.4f}"
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()

## 4. 5-fold cross-validation

In [ ]:
def train_one_fold(fold: int, train_df: pd.DataFrame, val_df: pd.DataFrame) -> dict:
    fold_dir = ARTIFACT_DIR / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    best_model_path = fold_dir / "best_model.pt"
    metrics_path    = fold_dir / "heldout_metrics.csv"
    logits_path     = fold_dir / "heldout_logits.npz"

    if SKIP_COMPLETED_FOLDS and metrics_path.exists() and logits_path.exists() and best_model_path.exists():
        print(f"Fold {fold}: already done — skipping.")
        return pd.read_csv(metrics_path).iloc[0].to_dict()

    seed = SEED + fold
    set_seed(seed)

    train_loader = make_loader(train_df, BATCH_SIZE, shuffle=True,  augment=True,  seed=seed)
    val_loader   = make_loader(val_df,   BATCH_SIZE, shuffle=False, augment=False, seed=seed)
    held_loader  = make_loader(heldout_df, BATCH_SIZE, shuffle=False, augment=False, seed=seed)

    model     = CNNBiLSTMAttention(IN_CHANNELS, len(CLASS_NAMES)).to(DEVICE)
    criterion = class_weighted_ce(train_df, "class_id", len(CLASS_NAMES))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LR)
    scaler    = torch.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    best_score, patience_ctr = -np.inf, 0
    history = []

    for epoch in tqdm(range(1, EPOCHS + 1), desc=f"fold {fold}"):
        train_loss, ty, tl, _ = run_epoch(model, train_loader, criterion, optimizer, scaler, True,  f"f{fold} train")
        val_loss,   vy, vl, _ = run_epoch(model, val_loader,   criterion, None,      None,  False, f"f{fold} val")
        scheduler.step()

        tp = stable_softmax(tl); vp = stable_softmax(vl)
        tm = metrics_from_probs(ty, tp, CLASS_NAMES, "train_")
        vm = metrics_from_probs(vy, vp, CLASS_NAMES, "val_")
        score = vm["val_macro_f1"]

        history.append({"fold": fold, "epoch": epoch,
                        "train_loss": train_loss, "val_loss": val_loss,
                        "checkpoint_score": score, **tm, **vm})
        tqdm.write(
            f"fold={fold} ep={epoch:03d} "
            f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
            f"val_f1={vm['val_macro_f1']:.4f} "
            f"val_auc={vm['val_macro_ovr_auc']:.4f}"
        )

        if score > best_score:
            best_score = score; patience_ctr = 0
            torch.save({"model_state_dict": model.state_dict(), "fold": fold,
                        "epoch": epoch, "best_score": best_score}, best_model_path)
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                tqdm.write(f"fold={fold} early stop at epoch {epoch}")
                break

    pd.DataFrame(history).to_csv(fold_dir / "training_history.csv", index=False)

    # Heldout evaluation with best model
    ckpt = torch.load(best_model_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    best_epoch = ckpt["epoch"]

    _, hy, hl, hs = run_epoch(model, held_loader, criterion, None, None, False, f"f{fold} heldout")
    hp = stable_softmax(hl)
    hm = metrics_from_probs(hy, hp, CLASS_NAMES, "heldout_")
    hm.update({"fold": fold, "heldout_loss": 0.0, "best_epoch": best_epoch, "best_score": best_score})

    pd.DataFrame([hm]).to_csv(metrics_path, index=False)
    np.savez(logits_path, y_true=hy, probs=hp, stems=hs)

    # Save val split for possible future use
    val_df.to_csv(fold_dir / "val_split.csv", index=False)
    train_df.to_csv(fold_dir / "train_split.csv", index=False)

    # Confusion matrix
    plot_confusion_matrix(hy, hp.argmax(axis=1), hp, fold,
                          hm["heldout_macro_f1"], hm["heldout_macro_ovr_auc"],
                          CLASS_NAMES, val_score=best_score)

    print(f"Fold {fold}: heldout F1={hm['heldout_macro_f1']:.4f}  "
          f"Bal={hm['heldout_balanced_accuracy']:.4f}  "
          f"AUC={hm['heldout_macro_ovr_auc']:.4f}  "
          f"best_ep={best_epoch}")
    return hm

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_metrics = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_val_df, train_val_df["class_id"])):
    train_df = train_val_df.iloc[train_idx].copy().reset_index(drop=True)
    val_df   = train_val_df.iloc[val_idx].copy().reset_index(drop=True)
    print(f"\n{'='*60}")
    print(f"Fold {fold}: train={len(train_df)}, val={len(val_df)}")
    m = train_one_fold(fold, train_df, val_df)
    fold_metrics.append(m)

metrics_df = pd.DataFrame(fold_metrics)
metrics_df.to_csv(ARTIFACT_DIR / "heldout_fold_metrics.csv", index=False)
print("\n=== Per-fold heldout F1 ===")
for f, m in enumerate(fold_metrics):
    print(f"  Fold {f}: F1={m['heldout_macro_f1']:.4f}  Bal={m['heldout_balanced_accuracy']:.4f}  "
          f"ep={m.get('best_epoch','?')}")
print(f"  Mean F1: {np.mean([m['heldout_macro_f1'] for m in fold_metrics]):.4f}")

## 5. 5-fold soft ensemble

In [ ]:
probs_list, y_ref, stems_ref = [], None, None

for fold in range(5):
    p = ARTIFACT_DIR / f"fold_{fold}" / "heldout_logits.npz"
    d = np.load(p, allow_pickle=False)
    if y_ref is None:
        y_ref, stems_ref = d["y_true"], d["stems"]
    else:
        assert np.array_equal(y_ref, d["y_true"]), f"Label mismatch fold {fold}"
    probs_list.append(d["probs"])

ensemble_probs = np.mean(probs_list, axis=0)
ensemble_pred  = ensemble_probs.argmax(axis=1)

ens_metrics = metrics_from_probs(y_ref, ensemble_probs, CLASS_NAMES, "heldout_")
ens_metrics["n_models"] = len(probs_list)
pd.DataFrame([ens_metrics]).to_csv(ARTIFACT_DIR / "heldout_ensemble_metrics.csv", index=False)

print("=== CNN-BiLSTM-Attention 5-fold Soft Ensemble ===")
print(f"  F1={ens_metrics['heldout_macro_f1']:.4f}  "
      f"Bal={ens_metrics['heldout_balanced_accuracy']:.4f}  "
      f"AUC={ens_metrics['heldout_macro_ovr_auc']:.4f}")
recalls = [ens_metrics[f"heldout_recall_{c}"] for c in CLASS_NAMES]
print(f"  Ctrl={recalls[0]:.3f}  MR={recalls[1]:.3f}  MS={recalls[2]:.3f}")
print()
print(classification_report(y_ref, ensemble_pred, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(y_ref, ensemble_pred, labels=[0,1,2])
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
cm_df = pd.DataFrame(cm_norm, index=CLASS_NAMES, columns=CLASS_NAMES)
print("Normalized confusion matrix:")
print(cm_df.round(3).to_string())

pd.DataFrame(cm,      index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(ARTIFACT_DIR / "heldout_confusion_matrix.csv")
pd.DataFrame(cm_norm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(ARTIFACT_DIR / "heldout_confusion_matrix_normalized.csv")